# Exploratory Data Analysis — AI Silent Disease Predictor

**Pipeline Overview**
- 5 clinical datasets (~80K samples)
- Unified schema: age, sex, blood_pressure, cholesterol, glucose, bmi, heart_rate, smoking, exercise
- 13 ML features (9 biomarkers + 4 interaction features)
- Target: binary health risk classification

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Project paths
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.path.basename(os.getcwd()) != 'notebooks':
    PROJECT_ROOT = os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')

sns.set_theme(style='whitegrid', palette='deep', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
print(f'Project root: {PROJECT_ROOT}')
print(f'Data dir: {DATA_DIR}')

## 1. Load Fused Dataset

Load the unified dataset created by `data/dataset_fusion.py`.

In [ ]:
fused_path = os.path.join(DATA_DIR, 'fused_dataset.csv')
if os.path.exists(fused_path):
    df_fused = pd.read_csv(fused_path)
    print(f'Fused dataset: {df_fused.shape[0]:,} rows × {df_fused.shape[1]} cols')
    display(df_fused.head())
    display(df_fused.describe())
else:
    print(f'⚠ {fused_path} not found. Run: python data/dataset_fusion.py')
    df_fused = None

## 2. Class Distribution

In [ ]:
if df_fused is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bar chart
    counts = df_fused['target'].value_counts().sort_index()
    ax = axes[0]
    bars = ax.bar(['Low Risk (0)', 'High Risk (1)'], counts.values,
                  color=['#4CAF50', '#F44336'], edgecolor='white', linewidth=1.5)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                f'{val:,}', ha='center', fontweight='bold', fontsize=12)
    ax.set_title('Target Class Distribution', fontweight='bold')
    ax.set_ylabel('Count')
    
    # Pie chart
    ax = axes[1]
    ax.pie(counts.values, labels=['Low Risk', 'High Risk'],
           autopct='%1.1f%%', colors=['#4CAF50', '#F44336'],
           startangle=90, explode=(0.02, 0.02))
    ax.set_title('Class Balance', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(os.path.join(DATA_DIR, '..', '..', 'visualization', 'class_distribution.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    
    ratio = counts.min() / counts.max()
    print(f'\nClass ratio: {ratio:.2f} ({"balanced" if ratio > 0.4 else "imbalanced — SMOTE needed"})')

## 3. Feature Distributions

In [ ]:
if df_fused is not None:
    numeric_cols = ['age', 'blood_pressure', 'cholesterol', 'glucose', 'bmi', 'heart_rate']
    available = [c for c in numeric_cols if c in df_fused.columns]
    
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.ravel()
    
    for i, col in enumerate(available):
        ax = axes[i]
        for label, color in [(0, '#4CAF50'), (1, '#F44336')]:
            subset = df_fused[df_fused['target'] == label][col].dropna()
            ax.hist(subset, bins=50, alpha=0.6, color=color,
                    label=f'Target={label}', density=True)
        ax.set_title(f'{col}', fontweight='bold')
        ax.legend(fontsize=9)
    
    for j in range(len(available), len(axes)):
        axes[j].set_visible(False)
    
    plt.suptitle('Feature Distributions by Target Class', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(DATA_DIR, '..', '..', 'visualization', 'feature_distributions.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

## 4. Correlation Matrix

In [ ]:
if df_fused is not None:
    corr = df_fused.select_dtypes(include=[np.number]).corr()
    
    fig, ax = plt.subplots(figsize=(12, 10))
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, vmin=-1, vmax=1, square=True, ax=ax,
                linewidths=0.5, cbar_kws={'shrink': 0.8})
    ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(DATA_DIR, '..', '..', 'visualization', 'correlation_matrix.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

## 5. Missing Values Analysis

In [ ]:
if df_fused is not None:
    missing = df_fused.isnull().sum()
    missing_pct = (missing / len(df_fused) * 100).round(2)
    missing_df = pd.DataFrame({'count': missing, 'pct': missing_pct})
    missing_df = missing_df[missing_df['count'] > 0].sort_values('pct', ascending=False)
    
    if len(missing_df) > 0:
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.barh(missing_df.index, missing_df['pct'], color='#FF7043')
        ax.set_xlabel('Missing %')
        ax.set_title('Missing Values by Feature', fontweight='bold')
        for i, (pct, count) in enumerate(zip(missing_df['pct'], missing_df['count'])):
            ax.text(pct + 0.3, i, f'{pct:.1f}% ({count:,})', va='center', fontsize=9)
        plt.tight_layout()
        plt.show()
    else:
        print('✓ No missing values in fused dataset')

## 6. Engineered Features Analysis

Load the 13 ML features (9 biomarkers + 4 interactions) from `features.csv`.

In [ ]:
features_path = os.path.join(DATA_DIR, 'features.csv')
if os.path.exists(features_path):
    df_feat = pd.read_csv(features_path)
    print(f'Features dataset: {df_feat.shape[0]:,} rows × {df_feat.shape[1]} cols')
    display(df_feat.head())
    display(df_feat.describe())
else:
    print(f'⚠ {features_path} not found. Run: python training/feature_engineering.py')
    df_feat = None

In [ ]:
if df_feat is not None:
    feature_cols = [c for c in df_feat.columns if c != 'target']
    
    corr_with_target = df_feat[feature_cols].corrwith(df_feat['target']).abs().sort_values(ascending=False)
    
    fig, ax = plt.subplots(figsize=(10, 7))
    bars = ax.barh(corr_with_target.index[::-1], corr_with_target.values[::-1],
                   color='#2196F3', edgecolor='#1565C0', linewidth=0.5)
    ax.set_xlabel('|Correlation with Target|')
    ax.set_title('Feature-Target Correlation (Absolute)', fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(DATA_DIR, '..', '..', 'visualization', 'feature_target_correlation.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

## 7. Biomarker Feature Distributions

In [ ]:
if df_feat is not None:
    biomarkers = ['face_fatigue', 'symmetry_score', 'blink_instability',
                  'brightness_variance', 'voice_stress', 'breathing_score',
                  'pitch_instability', 'face_risk_score', 'voice_risk_score']
    available_bio = [c for c in biomarkers if c in df_feat.columns]
    
    rows = (len(available_bio) + 2) // 3
    fig, axes = plt.subplots(rows, 3, figsize=(16, 4 * rows))
    axes = axes.ravel()
    
    for i, col in enumerate(available_bio):
        ax = axes[i]
        for label, color in [(0, '#4CAF50'), (1, '#F44336')]:
            subset = df_feat[df_feat['target'] == label][col].dropna()
            ax.hist(subset, bins=50, alpha=0.6, color=color,
                    label=f'Target={label}', density=True)
        ax.set_title(f'{col}', fontweight='bold', fontsize=10)
        ax.legend(fontsize=8)
    
    for j in range(len(available_bio), len(axes)):
        axes[j].set_visible(False)
    
    plt.suptitle('Biomarker Feature Distributions', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 8. Interaction Features

In [ ]:
if df_feat is not None:
    interactions = ['cardio_stress', 'metabolic_score', 'fatigue_stress', 'respiratory_variation']
    available_int = [c for c in interactions if c in df_feat.columns]
    
    if available_int:
        fig, axes = plt.subplots(1, len(available_int), figsize=(5 * len(available_int), 5))
        if len(available_int) == 1:
            axes = [axes]
        
        for ax, col in zip(axes, available_int):
            for label, color in [(0, '#4CAF50'), (1, '#F44336')]:
                subset = df_feat[df_feat['target'] == label][col].dropna()
                ax.hist(subset, bins=50, alpha=0.6, color=color,
                        label=f'Target={label}', density=True)
            ax.set_title(col, fontweight='bold')
            ax.legend(fontsize=9)
        
        plt.suptitle('Interaction Feature Distributions', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
    else:
        print('No interaction features found')

## 9. Summary Statistics

In [ ]:
print('='*60)
print('  DATASET SUMMARY')
print('='*60)
if df_fused is not None:
    print(f'  Fused dataset:       {df_fused.shape[0]:>8,} rows × {df_fused.shape[1]} cols')
    print(f'  Missing values:      {df_fused.isnull().sum().sum():>8,}')
    print(f'  Target=0 (Low Risk): {(df_fused["target"]==0).sum():>8,}')
    print(f'  Target=1 (High Risk):{(df_fused["target"]==1).sum():>8,}')
if df_feat is not None:
    print(f'\n  Feature dataset:     {df_feat.shape[0]:>8,} rows × {df_feat.shape[1]} cols')
    print(f'  Biomarker features:  {len([c for c in df_feat.columns if c != "target"]):>8}')
print('='*60)